# Circular-finned-tube dry-air example (v0.7.0)

This deterministic whole-exchanger example uses IAPWS liquid water inside a welded circular-finned tube bank and the built-in dry-air properties outside. It is deliberately within the published Briggs–Young and Robinson–Briggs applicability ranges. All quantities are SI. The outside film HTC and the extended-surface effective HTC are reported separately.

In [1]:
import math
from pathlib import Path
import sys

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'core').is_dir() and (path / 'pyproject.toml').is_file()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from core.geometry import BareTube, CircularFinnedTube, TubeBundle
from core.models import BareTubeHeatExchanger, BalanceSideSpec, HXSideInput
from core.phase_change.types import PhaseChangeMode
from core.properties.dry_air import DryAirPropertyProvider
from core.properties.water import IAPWS97WaterSteamProvider


In [2]:
# 6 x 8 staggered welded-finned bank; all stated dimensions are metres.
core = BareTube(
    D_i=0.021, D_o=0.025, length_total=2.0, length_effective=2.0, wall_k=45.0,
)
tube = CircularFinnedTube(
    core_tube=core, fin_k=200.0, D_fin=0.052, D_root=0.025,
    fin_thickness_root=0.0005, fin_thickness_tip=0.0005,
    fin_pitch=0.003,
    fin_contact_efficiency=1.0,  # 1.0 = ideal fin/tube thermal contact
)
pitch_transverse = 0.060
bundle = TubeBundle(
    tube=tube, n_rows=6, n_tubes_per_row=8,
    pitch_transverse=pitch_transverse,
    pitch_longitudinal=math.sqrt(3.0) * pitch_transverse / 2.0,
    layout='staggered', n_passes_tube=2, flow_arrangement='crossflow',
)
hx = BareTubeHeatExchanger(bundle)

water = IAPWS97WaterSteamProvider()
dry_air = DryAirPropertyProvider(prefer_coolprop=False)
inside = HXSideInput(
    provider=water, m_dot=1.5, T_in=360.0, p=1.0e6,
    phase_change_mode=PhaseChangeMode.DISABLED,
)
outside = HXSideInput(
    provider=dry_air, m_dot=3.6, T_in=300.0, p=101_325.0,
    phase_change_mode=PhaseChangeMode.DISABLED,
)

print(f'gross outside area: {bundle.total_outer_area:.6f} m²')
print(f'face / minimum free-flow area: {bundle.frontal_flow_area:.6f} / {bundle.minimum_free_flow_area:.6f} m²')
print(f'periodic blockage width: {bundle.projected_blocking_area_per_length:.6f} m')

gross outside area: 113.398928 m²
face / minimum free-flow area: 0.960000 / 0.488000 m²
periodic blockage width: 0.029500 m


In [3]:
simulation = hx.simulate(inside, outside)
diagnostics = simulation.finned_tube_diagnostics
assert diagnostics is not None

print('Dry circular-finned-tube Simulation')
print(f'Q: {simulation.q:.3f} W')
print(f'Tout water / air: {simulation.T_out_inside:.3f} / {simulation.T_out_outside:.3f} K')
print(f'outside_alpha_physical: {diagnostics.outside_alpha_physical:.6f} W/(m² K)')
print(f'outside_alpha_effective_gross: {diagnostics.outside_alpha_effective_gross:.6f} W/(m² K)')
print(f'eta_fin / eta_overall: {diagnostics.eta_fin:.8f} / {diagnostics.eta_overall:.8f}')
print(f'U / UA: {diagnostics.U:.6f} W/(m² K) / {diagnostics.UA:.6f} W/K')
print(f'dp outside: {diagnostics.dp:.6f} Pa')
print(f'Vface / Vmax / Re_Droot: {diagnostics.face_velocity:.6f} / {diagnostics.reference_velocity:.6f} m/s / {diagnostics.Re:.3f}')
print(f'correlations: {diagnostics.heat_transfer_metadata.method} / {diagnostics.pressure_drop_metadata.method}')
print(
    f'contact_input_mode: {diagnostics.contact_input_mode} '
    f'(fin_contact_efficiency_input={diagnostics.fin_contact_efficiency_input}, '
    f'effective={diagnostics.fin_contact_efficiency_effective:.8f})'
)
print(
    f'contact_resistance_equivalent_areal: {diagnostics.contact_resistance_equivalent_areal:.6e} m² K/W  '
    f'(resistance_contact: {diagnostics.resistance_contact:.6e} K/W)'
)

Dry circular-finned-tube Simulation
Q: 118213.974 W
Tout water / air: 341.198 / 332.572 K
outside_alpha_physical: 57.566832 W/(m² K)
outside_alpha_effective_gross: 52.436956 W/(m² K)
eta_fin / eta_overall: 0.90566123 / 0.91088833
U / UA: 33.334224 W/(m² K) / 3780.065308 W/K
dp outside: 137.025416 Pa
Vface / Vmax / Re_Droot: 3.359492 / 6.608836 m/s / 9595.511
correlations: briggs_young_1963 / robinson_briggs_1966
contact_input_mode: contact_efficiency (fin_contact_efficiency_input=1.0, effective=1.00000000)
contact_resistance_equivalent_areal: 0.000000e+00 m² K/W  (resistance_contact: 0.000000e+00 K/W)


In [4]:
rating = hx.rate(
    BalanceSideSpec(
        provider=water, p=1.0e6, m_dot=1.5, T_in=360.0,
        T_out=simulation.T_out_inside,
    ),
    BalanceSideSpec(
        provider=dry_air, p=101_325.0, m_dot=3.6, T_in=300.0,
        T_out=simulation.T_out_outside,
    ),
)
rating_diagnostics = rating.finned_tube_diagnostics
assert rating_diagnostics is not None

print('Matching Rating')
print(f'Q required / UA actual: {rating.Q_required:.3f} W / {rating.UA_actual:.6f} W/K')
print(f'outside_alpha_physical: {rating_diagnostics.outside_alpha_physical:.6f} W/(m² K)')
print(f'outside_alpha_effective_gross: {rating_diagnostics.outside_alpha_effective_gross:.6f} W/(m² K)')
print(f'U / UA / dp: {rating_diagnostics.U:.6f} W/(m² K) / {rating_diagnostics.UA:.6f} W/K / {rating_diagnostics.dp:.6f} Pa')

Matching Rating
Q required / UA actual: 118213.176 W / 3780.065308 W/K
outside_alpha_physical: 57.566832 W/(m² K)
outside_alpha_effective_gross: 52.436956 W/(m² K)
U / UA / dp: 33.334224 W/(m² K) / 3780.065308 W/K / 137.025416 Pa


The dry fin correlations use `D_root` and the periodic minimum free-flow area. `outside_alpha_physical` is the film coefficient for source/correlation comparison. `outside_alpha_effective_gross` includes the actual fin/root/contact resistance network and is the generic exchanger-side coefficient referenced to gross outside area. The circular-finned model supports dry outside air with inside single-phase or supported inside phase-change routes; wet or condensing flow on the finned outside is intentionally rejected.